In [17]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import EarlyStopping

import copy
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, 
    average_precision_score,
    roc_auc_score, 
    precision_score, 
    recall_score, 
    accuracy_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    log_loss
)

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(f"current working directory: {os.getcwd()}")

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"device: {device}")

current working directory: /Users/cbrown/code/cse_284
device: mps


# Import and Preprocess

In [18]:
# paths
geno_path = '/Users/cbrown/code/cse_284/data/processed/Effect allele count.csv'
pheno_path = "data/raw/GenRisk_ImputedData_CRC_Samples.txt"

# import
df_geno = pd.read_csv(geno_path)
df_geno = df_geno.set_index('IID')
df_pheno = pd.read_csv(pheno_path, sep='\t')
df_pheno = df_pheno.set_index('ID')
df_pheno = df_pheno.reindex(df_geno.index)

print(df_geno.shape)
print(df_pheno.shape)

(4295, 204)
(4295, 1)


# Helper Functions

In [19]:
def evaluate_model(model_name, y_true, y_pred, y_prob, cv_score, cv_scoring):
    """
    Calculates key classification metrics and returns them as a dictionary.
    """

    pr_auc = average_precision_score(y_true, y_prob, pos_label='Colorectal')
    roc_auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, pos_label='Colorectal')
    precision = precision_score(y_true, y_pred, pos_label='Colorectal')
    recall = recall_score(y_true, y_pred, pos_label='Colorectal')
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    ll = log_loss(y_true, y_prob)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    
    metrics_dict = {
        'Model': model_name,
        'PR_AUC': round(pr_auc, 4),
        'F1_Score': round(f1, 4),
        'ROC_AUC': round(roc_auc, 4),
        'Precision': round(precision, 4),
        'Recall_Sensitivity': round(recall, 4),
        'Specificity': round(specificity, 4),
        'Accuracy': round(accuracy, 4),
        'Balanced Accuracy': round(balanced_accuracy, 4),
        f'{cv_scoring} during CV': cv_score,
        'Log Loss': round(ll, 4)
    }
    
    return metrics_dict

# Generate train test split

In [20]:
# Generate X genotype matrix and y phenotype array
X = df_geno.iloc[:,5:].copy()
y = np.array(df_pheno.values).ravel()

In [21]:
# stratified train test split for 20% hold-out set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)
print(f"Outer split shape: {X_train_val.shape}")
print(f"Hold-out test set: {X_test.shape}")

Outer split shape: (3436, 199)
Hold-out test set: (859, 199)


In [22]:
model_results = []

# Logistic Regression Model

In [23]:
lr_model = LogisticRegression(penalty='elasticnet', class_weight='balanced', random_state=42, solver='saga', max_iter=500)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_distributions = {
    'l1_ratio': [0, 0.2, 0.4, 0.6, 0.8, 1],
    'C': [0.1, 0.5, 1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='balanced_accuracy',
    cv=cv_strategy,        # 5-fold stratified CV
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# fit model
random_search.fit(X_train_val, y_train_val)
print(f"Best Parameters Found: {random_search.best_params_}")
print(f"Best Cross-Validation Balanced Accuracy: {random_search.best_score_:.4f}")

# make predictions on hold-out data using best model
best_lr = random_search.best_estimator_
y_test_probs = best_lr.predict_proba(X_test)[:, 1]
y_test_preds = best_lr.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_test_preds, target_names=['Control', 'Case']))

top_5_snps = pd.Series(best_lr.coef_.ravel(), index=X.columns).nlargest(5)
print("\n--- Top 5 Most Important SNPs ---")
print(top_5_snps)

model_results.append(evaluate_model('Logistic Regression', y_test, y_test_preds, y_test_probs, random_search.best_score_, 'Balanced Accuracy'))

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

Best Parameters Found: {'l1_ratio': 1, 'C': 0.1}
Best Cross-Validation Balanced Accuracy: 0.5564
Classification Report:
              precision    recall  f1-score   support

     Control       0.39      0.53      0.45       286
        Case       0.72      0.59      0.65       573

    accuracy                           0.57       859
   macro avg       0.56      0.56      0.55       859
weighted avg       0.61      0.57      0.58       859


--- Top 5 Most Important SNPs ---
8:117630683_A_C_A     0.232646
12:4388271_C_T_C      0.227947
20:6699595_T_G_T      0.151307
10:80819132_A_G_A     0.143048
10:101344263_G_A_G    0.131278
dtype: float64


# Random Forest Model

In [24]:
# Model
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_distributions = {
    'n_estimators': [300, 500, 800, 1000],
    'max_features': ['sqrt', 'log2', 20, 50],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10]
}

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='balanced_accuracy',
    cv=cv_strategy,        # 5-fold stratified CV
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# fit model
random_search.fit(X_train_val, y_train_val)
print(f"Best Parameters Found: {random_search.best_params_}")
print(f"Best Cross-Validation Balanced Accuracy: {random_search.best_score_:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_test_preds, target_names=['Control', 'Case']))

# get predictions on hold out set using best model
best_rf = random_search.best_estimator_
y_test_probs = best_rf.predict_proba(X_test)[:, 1]
y_test_preds = best_rf.predict(X_test)

feature_importances = pd.Series(best_rf.feature_importances_, index=X.columns)
top_5_snps = feature_importances.nlargest(5)
print("\n--- Top 5 Most Important SNPs ---")
print(top_5_snps)

model_results.append(evaluate_model('Random Forest', y_test, y_test_preds, y_test_probs, random_search.best_score_, 'Balanced Accuracy'))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters Found: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 5}
Best Cross-Validation Balanced Accuracy: 0.5382
Classification Report:
              precision    recall  f1-score   support

     Control       0.39      0.53      0.45       286
        Case       0.72      0.59      0.65       573

    accuracy                           0.57       859
   macro avg       0.56      0.56      0.55       859
weighted avg       0.61      0.57      0.58       859


--- Top 5 Most Important SNPs ---
18:46452327_A_G_A    0.015354
20:47340117_A_G_A    0.014204
12:51216890_T_C_T    0.012162
2:219191256_T_C_T    0.009944
16:68743939_A_C_A    0.009850
dtype: float64


# NN Model

In [25]:
X_train_val_np = X_train_val.values.astype(np.float32)
y_train_val_np = np.asarray([1 if x == 'Colorectal' else 0 for x in y_train_val], dtype=np.float32)

X_test_np = X_test.values.astype(np.float32)

# model
class PRSPredictor(nn.Module):
    def __init__(self, input_dim=199, hidden_1=64, hidden_2=32, dropout_rate=0.4):
        super(PRSPredictor, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.BatchNorm1d(hidden_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_1, hidden_2),
            nn.BatchNorm1d(hidden_2),
            nn.ReLU(),
            nn.Dropout(max(0, dropout_rate - 0.1)) # Keep second dropout slightly lower
        )
        
        self.output = nn.Linear(hidden_2, 1)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)
        return x

# balance class weights
num_negatives = (y_train_val_np == 0).sum()
num_positives = (y_train_val_np == 1).sum()
pos_weight = torch.tensor([num_negatives / num_positives], dtype=torch.float32)

# early stopping
early_stopping = EarlyStopping(
    monitor='valid_loss', 
    patience=15, 
    lower_is_better=True
)

# wrap model in skorch
net = NeuralNetBinaryClassifier(
    module=PRSPredictor,
    module__input_dim=199,
    criterion=nn.BCEWithLogitsLoss,
    criterion__pos_weight=pos_weight,
    optimizer=torch.optim.Adam,
    max_epochs=100,
    callbacks=[early_stopping],
    verbose=0,
    device=device
)

param_grid = {
    'module__hidden_1': [32, 64, 128],
    'module__hidden_2': [16, 32, 64],
    'module__dropout_rate': [0.3, 0.4, 0.5],
    'lr': [0.0005, 0.001, 0.005],
    'batch_size': [16, 32, 64]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting randomize search for NN with 5-fold CV")

random_search = RandomizedSearchCV(
    estimator=net,
    param_distributions=param_grid,
    n_iter=15,
    scoring='balanced_accuracy',
    cv=cv_strategy,
    n_jobs=1,
    verbose=2,
    random_state=42
)

# fit model on training data
random_search.fit(X_train_val_np, y_train_val_np)
print(f"Best Parameters Found: {random_search.best_params_}")
print(f"Best CV Bal. Acc. Score: {random_search.best_score_:.4f}")

# make predictions on hold-out data using best model
best_nn = random_search.best_estimator_
y_test_probs = best_nn.predict_proba(X_test_np)[:, 1]
y_test_preds = best_nn.predict(X_test_np)
y_test_preds = ['Colorectal' if x == 1 else 'Control' for x in y_test_preds]

model_results.append(
    evaluate_model('Tuned PyTorch NN', 
        y_test, 
        y_test_preds, 
        y_test_probs, 
        random_search.best_score_, 
        'Balanced Accuracy'
    )
)

Starting randomize search for NN with 5-fold CV
Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.5, module__hidden_1=128, module__hidden_2=16; total time=  10.9s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.5, module__hidden_1=128, module__hidden_2=16; total time=  11.9s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.5, module__hidden_1=128, module__hidden_2=16; total time=  11.9s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.5, module__hidden_1=128, module__hidden_2=16; total time=  10.7s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.5, module__hidden_1=128, module__hidden_2=16; total time=  11.3s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.3, module__hidden_1=128, module__hidden_2=16; total time=  11.2s
[CV] END batch_size=16, lr=0.0005, module__dropout_rate=0.3, module__hidden_1=128, module__hidden_2=16; total time=  10.6s
[CV] END batch_size=16, lr=0.0

In [26]:
results_df = pd.DataFrame(model_results)
results_df = results_df.sort_values(by='PR_AUC', ascending=False)
print(results_df.to_string(index=False))

results_df.to_csv('results/tables/prs_model_comparison_metrics.csv', index=False)

              Model  PR_AUC  F1_Score  ROC_AUC  Precision  Recall_Sensitivity  Specificity  Accuracy  Balanced Accuracy  Balanced Accuracy during CV  Log Loss
   Tuned PyTorch NN  0.3827    0.4294   0.4446     0.3706              0.5105       0.5105    0.5483             0.5388                     0.552921    1.2466
Logistic Regression  0.2973    0.4531   0.5712     0.3948              0.5315       0.5315    0.5728             0.5624                     0.556417    0.6930
      Random Forest  0.2952    0.3361   0.5660     0.4133              0.2832       0.2832    0.6275             0.5413                     0.538247    0.6835
